# Experimentos #104 — Parte 2: pista Sonnet, grilla de hiperparámetros, épocas y arreglo de la pérdida

Orquestador **reanudable** (misma lógica que el notebook 04, que queda como registro de la Parte 1). Usa el mismo
experimento MLflow (`clasificacion-funcion-104`), así los runs de la Parte 1 con la misma configuración se reanudan.

- **Pista M («mixta»)**: entrenamiento D1–D6 (QLoRA, jurado, Haiku) y V con la mejor etiqueta disponible.
- **Pista S («sonnet»)**: entrenamiento D7–D9 y V con etiquetas Sonnet (mismas citas, otras etiquetas).
- Dentro de una pista, el early stopping y la elección usan **su** `val`. Entre pistas, `val` no es comparable: todo run
  se evalúa además en `val_mixta`, `val_sonnet`, `dev_haiku` y `ref_gpt`.
- Los runs `kd`/`ponderado` de la Parte 1 eran copias del duro (bug de `preparar_ejemplos`): se invalidan y se re-corren.

> **Provisional**: no hay Test Gold humano. La comparación final entre pistas espera al Test Gold.

Primero `DRY = True` (64 casos, 1 época, experimento `-dry`); luego `DRY = False`.

In [1]:
import dataclasses
import gc
import json
import re
import shutil
import statistics
import sys
import time
import traceback
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "packages/clasificacion-funcion/src"))
sys.path.insert(0, str(REPO / "packages/auditoria-corpus/src"))
sys.path.insert(0, str(REPO / "packages/recuperacion-densa/src"))
sys.path.insert(0, str(REPO))  # `src.tracking` y `proyecto_de_grado` (namespace desde la raiz)

import mlflow
import pandas as pd
import torch

from clasificacion_funcion import adaptador_proyecto_grado as ad
from clasificacion_funcion import anotacion_sonnet as an
from clasificacion_funcion import clasificador_torch as ct
from clasificacion_funcion import dominio
from clasificacion_funcion import experimento_104 as ex
from clasificacion_funcion import registro_mlflow as rm
from clasificacion_funcion import seleccion_datos as sd

/home/carlos/microproyecto-local-citation/local_workflows/.venv-qlora/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DRY = False          # True: 64 casos y 1 epoca por run, experimento y carpetas aparte
HORAS_MAX = 10.0     # se omiten las fases restantes al superarlo
SEMILLAS_EXTRA = (1234, 2026)
GRID_LR = (1e-5, 3e-5, 5e-5)
GRID_EPOCAS = (4, 10)
PACIENCIA_GRID = 3   # la linea base (lr 2e-5, 6 epocas, paciencia 2) ya existe
N_BOOTSTRAP = 200 if DRY else 1000

SUFIJO = "-dry" if DRY else ""
ARTIFACTS = REPO / f"proyecto_de_grado/artifacts/experimentos_104{'_dry' if DRY else ''}"
EVIDENCIAS = REPO / f"proyecto_de_grado/docs/evidencias_experimentos_104{'_dry' if DRY else ''}"
CORRIDAS = ARTIFACTS / "corridas"
T0 = time.time()
FALLAS = []

rm.configurar(rm.EXPERIMENTO + SUFIJO)
CLIENTE = mlflow.MlflowClient()
print("tracking:", mlflow.get_tracking_uri(), "| DRY =", DRY)


def hay_tiempo() -> bool:
    return (time.time() - T0) / 3600 < HORAS_MAX


def liberar_gpu() -> None:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def correr(nombre, fn, *a, **kw):
    """Ejecuta un run sin detener el notebook: la falla queda FAILED en MLflow y en FALLAS.
    Ante `OutOfMemoryError` reintenta una vez con micro-lote 4 y acumulacion 4 (mismo lote efectivo,
    queda en los params del run)."""
    if not hay_tiempo():
        print(f"[omitido por HORAS_MAX] {nombre}")
        return None
    t = time.time()
    for intento in (1, 2):
        try:
            run_id = fn(nombre, *a, **kw)
            print(f"[ok {time.time() - t:6.0f}s] {nombre} -> {run_id}")
            return run_id
        except torch.OutOfMemoryError:
            print(f"[OOM] {nombre} (intento {intento})")
            if intento == 1 and "hp" in kw:
                kw = {**kw, "hp": dataclasses.replace(kw["hp"], batch_size=4, acumulacion=4)}
                continue
            FALLAS.append(nombre)
            return None
        except Exception:
            FALLAS.append(nombre)
            print(f"[FALLO] {nombre}\n{traceback.format_exc()}")
            return None
        finally:
            liberar_gpu()


def puntaje(run_id):
    """Macro-F1 en la `val` PROPIA de la pista (criterio de eleccion; ya no dev_haiku + ref_gpt)."""
    if not run_id:
        return -1.0
    return mlflow.get_run(run_id).data.metrics.get("val_f1_macro", -1.0)


def runs_por_nombre(nombre, experimento=rm.EXPERIMENTO):
    """Runs de un experimento cuyo nombre (sin marca de tiempo) es `nombre`, el mas reciente primero."""
    exp = mlflow.get_experiment_by_name(experimento)
    if exp is None:
        return []
    runs = mlflow.search_runs([exp.experiment_id], output_format="list", order_by=["attributes.start_time DESC"])
    return [r for r in runs if r.data.tags.get("mlflow.runName", "").rsplit("-", 2)[0] == nombre]

tracking: sqlite:////home/carlos/microproyecto-local-citation/mlflow.db | DRY = False


## Fase 0 — Datos (P, V, D1–D9, V_sonnet), fugas, manifiesto e invalidación de runs de la Parte 1

In [3]:
# 0.1 Etiquetas Sonnet consolidadas (reanudable: solo une las salidas validas de cada lote)
print(an.consolidar(REPO / "proyecto_de_grado/anotacion/sonnet/salidas", ad.ETIQUETAS_SONNET), "filas Sonnet ->", ad.ETIQUETAS_SONNET)

FUENTES = ad.cargar_fuentes_104()
PAPERS = ad.cargar_papers()
print("citas con etiqueta Sonnet (Evaluable):", len(FUENTES.sonnet))
RES = sd.construir_todo(FUENTES)          # lanza ValueError si hay cualquier fuga
V, V_SONNET, VARIANTES = RES["v"], RES["v_sonnet"], RES["variantes"]

# 0.2 Manifiesto: el de la Parte 1 se conserva; su sha entra en la config de los runs de la pista M
ACTUAL = EVIDENCIAS / "manifiesto_datos.json"
PARTE1 = EVIDENCIAS / "manifiesto_datos_parte1.json"
if not DRY and not PARTE1.exists() and ACTUAL.exists():
    shutil.copy2(ACTUAL, PARTE1)
MANIFIESTO = sd.escribir_artefactos(RES, ARTIFACTS / "datos", EVIDENCIAS, {
    "qlora_18000_v2": ad.PREDS_QLORA[0], "jurado": ad.JURADO, "referencia_gpt": ad.REFERENCIA_GPT,
    "haiku_2700": ad.HAIKU_2700, "split_v2": ad.SPLIT_V2, "sonnet": ad.ETIQUETAS_SONNET})
SHA_DATOS = sd.sha_archivo(ACTUAL)
# Pista M: la MISMA config que en la Parte 1 (datos_sha de entonces) => sus runs se reanudan
SHA_DATOS_M = sd.sha_archivo(PARTE1) if PARTE1.exists() else SHA_DATOS
AUDITORIA = sd.auditar_fuentes(FUENTES)
(EVIDENCIAS / "auditoria_fuentes.json").write_text(json.dumps(AUDITORIA, ensure_ascii=False, indent=2), encoding="utf-8")

print("pool:", RES["info_pool"])
print("fugas:", RES["fugas"]["total_fugas"], "| citas de anios excluidos:", RES["fugas"]["citas_de_anios_excluidos"])
assert RES["fugas"]["total_fugas"] == 0 and RES["fugas"]["citas_de_anios_excluidos"] == 0
for n, c in MANIFIESTO["conteos"].items():
    print(f"{n:17s} n={MANIFIESTO['n'][n]:6d}", list(c.values()))

COMUN_M = {"v_sha": MANIFIESTO["sha256_ids"]["validacion"], "datos_sha": SHA_DATOS_M}
COMUN_S = {"v_sha": MANIFIESTO["sha256_ids"]["validacion_sonnet"], "datos_sha": SHA_DATOS,
           "pista": "sonnet", "v_etiquetas": "sonnet"}

with rm.abrir_corrida("auditoria-datos-part2", {"fase": "p2-0", "sha_datos": SHA_DATOS, "dry": DRY}, {"manifiesto_datos_sha": SHA_DATOS}) as run:
    if run is not None:
        mlflow.log_metric("fugas_total", RES["fugas"]["total_fugas"])
        mlflow.log_metrics({f"n_{n}": v for n, v in MANIFIESTO["n"].items()})
        mlflow.log_dict(MANIFIESTO, "manifiesto_datos.json")

16711 filas Sonnet -> /home/carlos/microproyecto-local-citation/proyecto_de_grado/anotacion/sonnet/etiquetas_sonnet.csv
citas con etiqueta Sonnet (Evaluable): 16704
pool: {'pool': 40487, 'descartes': {'sin_etiqueta_confiable': 3398, 'qlora_abstiene': 664, 'haiku_abstiene': 58, 'id_vetado': 12, 'anio_excluido': 19095}}
fugas: 0 | citas de anios excluidos: 0
qlora_p50         n= 13332 [2000, 1212, 1613, 2000, 2000, 656, 2000, 970, 881]
qlora_p80         n=  9826 [2000, 227, 568, 2000, 2000, 314, 1414, 613, 690]
jurado            n=  8887 [2000, 683, 379, 1433, 2000, 771, 1386, 85, 150]
hibrido           n= 13719 [2000, 1449, 1505, 2000, 2000, 1031, 1998, 895, 841]
natural           n= 13331 [5354, 448, 600, 1410, 3698, 235, 905, 358, 323]
haiku_solo        n=  2180 [844, 76, 120, 262, 586, 48, 137, 60, 47]
sonnet            n= 13325 [4214, 686, 698, 2286, 2023, 188, 2164, 322, 744]
sonnet_alta       n=  9066 [2906, 289, 276, 1864, 1766, 165, 1078, 217, 505]
sonnet_cap        n= 10638 [20

In [4]:
# 0.3 Invalidar los runs de la Parte 1 afectados (kd/ponderado nunca aplicaron su perdida; el TF-IDF se re-corre
#     para tener los conjuntos de evaluacion nuevos). Quedan KILLED + tag `invalido`: no se reanudan.
INVALIDOS = {"scibert-A-natural-kd-s42": "perdida_kd_no_aplicada_bug_preparar_ejemplos",
             "scibert-A-natural-ponderado-s42": "perdida_ponderada_no_aplicada_bug_preparar_ejemplos",
             "tfidf-lr-d1": "rerun_con_val_mixta_y_val_sonnet"}
if not DRY:
    for nombre, motivo in INVALIDOS.items():
        for r in runs_por_nombre(nombre):
            if r.info.status == "FINISHED" and "invalido" not in r.data.tags:
                rm.invalidar(r.info.run_id, motivo)
                print("invalidado:", nombre, r.info.run_id[:8])

invalidado: scibert-A-natural-kd-s42 244b433e
invalidado: scibert-A-natural-ponderado-s42 930c90ac
invalidado: tfidf-lr-d1 2b1a3594


## Fase 1 — Re-evaluar los checkpoints de la Parte 1 en `val_mixta` / `val_sonnet` (sin reentrenar)

In [5]:
# Los 10 checkpoints distintos de la Parte 1 (kd y ponderado eran copias de natural y no tienen checkpoint propio util)
PARTE1_CKPT = [("scibert-A-qlora_p50-s42", "scibert"), ("scibert-A-qlora_p80-s42", "scibert"),
               ("scibert-A-jurado-s42", "scibert"), ("scibert-A-hibrido-s42", "scibert"),
               ("scibert-A-natural-s42", "scibert"), ("scibert-A-haiku_solo-s42", "scibert"),
               ("scibert-A-natural-sinpesos-s42", "scibert"), ("scibert-A-natural-sinpesos-s1234", "scibert"),
               ("scibert-A-natural-sinpesos-s2026", "scibert"), ("specter-A-natural-sinpesos-s42", "specter")]
if DRY:  # en DRY solo uno (con la V completa): una V truncada registraria metricas falsas en un run real
    PARTE1_CKPT = PARTE1_CKPT[4:5]
REEVAL = {}
for nombre, encoder in PARTE1_CKPT:
    if not hay_tiempo():
        break
    runs = [r for r in runs_por_nombre(nombre) if r.info.status == "FINISHED" and "invalido" not in r.data.tags]
    carpeta = REPO / "proyecto_de_grado/artifacts/experimentos_104/corridas" / nombre
    if not runs or not (carpeta / "model.safetensors").is_file():
        print("sin run/checkpoint de la Parte 1:", nombre)
        continue
    try:
        REEVAL[nombre] = ex.reevaluar_checkpoint(runs[0].info.run_id, carpeta, encoder, V, FUENTES, PAPERS)
        print(f"[reeval] {nombre}: agregados {REEVAL[nombre] or 'nada (ya estaban)'}")
    except Exception:
        FALLAS.append(f"reeval-{nombre}")
        print(f"[FALLO] reeval {nombre}\n{traceback.format_exc()}")
    finally:
        liberar_gpu()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 25047.76it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[reeval] scibert-A-qlora_p50-s42: agregados ['val_mixta', 'val_sonnet']


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17082.82it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[reeval] scibert-A-qlora_p80-s42: agregados ['val_mixta', 'val_sonnet']


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 51282.04it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[reeval] scibert-A-jurado-s42: agregados ['val_mixta', 'val_sonnet']


... (10 outputs left out)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21062.01it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[reeval] scibert-A-natural-sinpesos-s2026: agregados ['val_mixta', 'val_sonnet']


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 42381.77it/s]


[reeval] specter-A-natural-sinpesos-s42: agregados ['val_mixta', 'val_sonnet']


## Orquestación: `sci()` con pista e hiperparámetros

In [6]:
V_PISTA = {"mixta": V, "sonnet": V_SONNET}


def sci(nombre, variante, semilla=42, modo="duro", pesos_clase=True, encoder="scibert", pista="mixta", **hp_kw):
    """Un run de SciBERT/SPECTER. Pista M con hiperparametros por defecto == config de la Parte 1 (se reanuda).
    Pista S agrega `pista` y `v_etiquetas` a la config. `hp_kw`: lr, epocas, paciencia, ..."""
    comun = COMUN_S if pista == "sonnet" else COMUN_M
    cfg = {**comun, "variante": variante, "ids_train_sha": MANIFIESTO["sha256_ids"][variante]}
    hp = ct.Hiperparametros(pesos_clase=pesos_clase, **hp_kw)
    return correr(nombre, ex.entrenar_y_registrar, cfg, VARIANTES[variante], V_PISTA[pista], FUENTES, PAPERS, CORRIDAS,
                  encoder=encoder, semilla=semilla, modo=modo, hp=hp, dry=DRY, manifiesto_datos_sha=SHA_DATOS)


def candidato(pista, stem, variante, modo="duro", **hp_kw):
    """Corre el run de semilla 42 y devuelve el registro con el que se elige el ganador de la pista."""
    nombre = f"{stem}-s42"
    run = sci(nombre, variante, modo=modo, pista=pista, **hp_kw)
    return {"pista": pista, "stem": stem, "nombre": nombre, "variante": variante, "modo": modo, "hp": hp_kw,
            "run": run, "val": puntaje(run)}


def ganador(candidatos):
    return max(candidatos, key=lambda c: c["val"])


def mostrar(candidatos):
    for c in sorted(candidatos, key=lambda c: -c["val"]):
        print(f"{c['val']:.4f}  {c['nombre']}")

## Fase 2 — Pista M: TF-IDF (re-corrida), y `kd` / `ponderado` reales sobre `natural` (s42)

In [7]:
D1 = VARIANTES["qlora_p50"]
correr("tfidf-lr-d1", ex.linea_base_tfidf, {**COMUN_M, "variante": "qlora_p50"}, D1, V, FUENTES, PAPERS, dry=DRY)

CAND_M = [candidato("mixta", "scibert-A-natural", "natural"),   # linea base de la Parte 1 (se reanuda)
          candidato("mixta", "scibert-A-natural-kd", "natural", modo="kd"),
          candidato("mixta", "scibert-A-natural-ponderado", "natural", modo="ponderado")]
mostrar(CAND_M)

[ok     33s] tfidf-lr-d1 -> 07a11726f8a34d2cacb2f1d23e296829
[ok      0s] scibert-A-natural-s42 -> f5ed045922b744cdb5a1fba68a279be5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 44057.35it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok   1213s] scibert-A-natural-kd-s42 -> 1b4aef7089284cf2984187a739dc0505


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 25779.61it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok   1214s] scibert-A-natural-ponderado-s42 -> db7592c47a414c4fbf6897a694335d22
0.7079  scibert-A-natural-kd-s42
0.6967  scibert-A-natural-ponderado-s42
0.6945  scibert-A-natural-s42


## Fase 3 — Pista S: TF-IDF y SciBERT sobre D7 `sonnet`, D8 `sonnet_alta`, D9 `sonnet_cap` (s42)

In [8]:
correr("tfidf-lr-sonnet", ex.linea_base_tfidf, {**COMUN_S, "variante": "sonnet"}, VARIANTES["sonnet"], V_SONNET, FUENTES, PAPERS, dry=DRY)

# Con etiquetas Sonnet (duras) kd == one-hot y ponderado == peso 1: la pista S solo usa perdida dura.
CAND_S = [candidato("sonnet", f"scibert-S-{v}", v) for v in sd.VARIANTES_SONNET]
mostrar(CAND_S)

[ok     31s] tfidf-lr-sonnet -> 3ce68e06f5744158b569fc6c84533091


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 30482.31it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok   1214s] scibert-S-sonnet-s42 -> 672d7f0aa4b244deb80fce8376a8f41b


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18240.49it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    843s] scibert-S-sonnet_alta-s42 -> f7694630599d4bdb8c6f5927342b2210


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 32946.49it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    834s] scibert-S-sonnet_cap-s42 -> fdf80ca04d864e1ab1ead94e5ae30d9e
0.6566  scibert-S-sonnet-s42
0.6468  scibert-S-sonnet_alta-s42
0.6282  scibert-S-sonnet_cap-s42


## Fase 4 — Grilla sobre la mejor configuración de cada pista

lr ∈ {1e-5, 3e-5, 5e-5} × épocas máximas ∈ {4, 10}, paciencia 3. La línea base (2e-5, 6 épocas) ya está entre los
candidatos. Las pistas se alternan en cada punto de la grilla: si se corta por tiempo quedan ambas parcialmente cubiertas.
Micro-lote (8) y `max_len` (512) **no cambian** (determinan la memoria por paso).

In [9]:
BASE = {"mixta": ganador(CAND_M), "sonnet": ganador(CAND_S)}
print({p: b["nombre"] for p, b in BASE.items()})
GRILLA = {"mixta": [BASE["mixta"]], "sonnet": [BASE["sonnet"]]}
for lr in GRID_LR:
    for ep in GRID_EPOCAS:
        for pista in ("mixta", "sonnet"):
            b = BASE[pista]
            if b["val"] < 0:
                continue
            GRILLA[pista].append(candidato(pista, f"{b['stem']}-lr{lr:g}-e{ep}", b["variante"], modo=b["modo"],
                                           lr=lr, epocas=ep, paciencia=PACIENCIA_GRID))
for pista, cands in GRILLA.items():
    print(f"-- {pista}")
    mostrar(cands)

{'mixta': 'scibert-A-natural-kd-s42', 'sonnet': 'scibert-S-sonnet-s42'}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 40429.47it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    822s] scibert-A-natural-kd-lr1e-05-e4-s42 -> f8997202f81d47199268a2731277973f


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 20880.76it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    829s] scibert-S-sonnet-lr1e-05-e4-s42 -> 261311372ad447a395f0f55553c2a769


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18752.34it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

... (15 outputs left out)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 29033.90it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok   1972s] scibert-A-natural-kd-lr5e-05-e10-s42 -> 40a5f9c6ba7d4742af860b12971f57a5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 35213.54it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok   1991s] scibert-S-sonnet-lr5e-05-e10-s42 -> f8170b85ffe44e828f000bf15658b850
-- mixta
0.7167  scibert-A-natural-kd-lr3e-05-e4-s42
0.7079  scibert-A-natural-kd-s42
0.7069  scibert-A-natural-kd-lr1e-05-e10-s42
0.7022  scibert-A-natural-kd-lr1e-05-e4-s42
0.7016  scibert-A-natural-kd-lr5e-05-e4-s42
0.6988  scibert-A-natural-kd-lr5e-05-e10-s42
0.6971  scibert-A-natural-kd-lr3e-05-e10-s42
-- sonnet
0.6736  scibert-S-sonnet-lr3e-05-e4-s42
0.6632  scibert-S-sonnet-lr1e-05-e10-s42
0.6566  scibert-S-sonnet-s42
0.6564  scibert-S-sonnet-lr5e-05-e10-s42
0.6536  scibert-S-sonnet-lr3e-05-e10-s42
0.6524  scibert-S-sonnet-lr5e-05-e4-s42
0.6509  scibert-S-sonnet-lr1e-05-e4-s42


## Fase 5 — Semillas 1234 y 2026 para la mejor configuración de cada pista

In [10]:
MEJOR = {p: ganador(c) for p, c in GRILLA.items()}
SEMILLAS = {}
for pista, b in MEJOR.items():
    SEMILLAS[pista] = {42: b["run"]}
for s in SEMILLAS_EXTRA:
    for pista, b in MEJOR.items():
        if b["val"] >= 0:
            SEMILLAS[pista][s] = sci(f"{b['stem']}-s{s}", b["variante"], semilla=s, modo=b["modo"], pista=pista, **b["hp"])
for pista, por_semilla in SEMILLAS.items():
    print(f"-- {pista}: {MEJOR[pista]['nombre']}")
    for metrica in ("val_f1_macro", "val_mixta_f1_macro", "val_sonnet_f1_macro", "dev_haiku_f1_macro", "ref_gpt_f1_macro"):
        xs = [mlflow.get_run(r).data.metrics.get(metrica) for r in por_semilla.values() if r]
        xs = [x for x in xs if x is not None]
        if len(xs) > 1:
            print(f"  {metrica}: {statistics.mean(xs):.4f} +- {statistics.stdev(xs):.4f} (n={len(xs)})")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 63617.87it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    821s] scibert-A-natural-kd-lr3e-05-e4-s1234 -> 9480b8db37734ec79f4870f50719ae9c


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 26280.43it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    827s] scibert-S-sonnet-lr3e-05-e4-s1234 -> d39abec5a38b4659927d9fabfb23bf87


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 25246.26it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    825s] scibert-A-natural-kd-lr3e-05-e4-s2026 -> 552e9a0f66fd4cd99c14521b42dd7b71


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 50269.00it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

[ok    828s] scibert-S-sonnet-lr3e-05-e4-s2026 -> 69b903729041413f90f1878e3aeacd4d
-- mixta: scibert-A-natural-kd-lr3e-05-e4-s42
  val_f1_macro: 0.7113 +- 0.0065 (n=3)
  val_mixta_f1_macro: 0.7113 +- 0.0065 (n=3)
  val_sonnet_f1_macro: 0.5009 +- 0.0075 (n=3)
  dev_haiku_f1_macro: 0.6338 +- 0.0312 (n=3)
  ref_gpt_f1_macro: 0.4971 +- 0.0192 (n=3)
-- sonnet: scibert-S-sonnet-lr3e-05-e4-s42
  val_f1_macro: 0.6667 +- 0.0133 (n=3)
  val_mixta_f1_macro: 0.5405 +- 0.0084 (n=3)
  val_sonnet_f1_macro: 0.6667 +- 0.0133 (n=3)
  dev_haiku_f1_macro: 0.5163 +- 0.0117 (n=3)
  ref_gpt_f1_macro: 0.6296 +- 0.0276 (n=3)


## Fase 6 — SPECTER con la mejor configuración de cada pista

In [11]:
R_SPECTER = {}
for pista, b in MEJOR.items():
    if b["val"] >= 0:
        R_SPECTER[pista] = sci(f"{b['stem'].replace('scibert', 'specter')}-s42", b["variante"], modo=b["modo"],
                               encoder="specter", pista=pista, **b["hp"])
print(R_SPECTER)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17940.56it/s]


[ok    834s] specter-A-natural-kd-lr3e-05-e4-s42 -> 0697cbc689624ca7b224ba75519f3b84


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21209.73it/s]


[ok    836s] specter-S-sonnet-lr3e-05-e4-s42 -> 284b0d2ce1434683a93310526449a5ff
{'mixta': '0697cbc689624ca7b224ba75519f3b84', 'sonnet': '284b0d2ce1434683a93310526449a5ff'}


## Fase 7 — Smoke test de inferencia de los 2 ganadores

In [12]:
SMOKE = []
for pista, b in MEJOR.items():
    carpeta = CORRIDAS / b["nombre"]
    if not (carpeta / "model.safetensors").is_file():
        carpeta = REPO / "proyecto_de_grado/artifacts/experimentos_104/corridas" / b["nombre"]  # runs reanudados de la Parte 1
    if not (carpeta / "model.safetensors").is_file():
        print("sin checkpoint:", b["nombre"])
        continue
    try:
        r = ex.smoke_inferencia(carpeta, "scibert", V_PISTA[pista], FUENTES, PAPERS)
    except Exception:
        FALLAS.append(f"smoke-{pista}")
        r = {"pista": pista, "ok": False, "error": traceback.format_exc()}
    SMOKE.append({"pista": pista, **r})
    print(SMOKE[-1])
    liberar_gpu()
(EVIDENCIAS / "smoke_inferencia_part2.json").write_text(json.dumps(SMOKE, ensure_ascii=False, indent=2), encoding="utf-8")
with rm.abrir_corrida("smoke-inferencia-part2", {"fase": "p2-7", "mejor": {p: b["nombre"] for p, b in MEJOR.items()}, "dry": DRY}) as run:
    if run is not None:
        for r in SMOKE:
            mlflow.log_metric(f"smoke_ok_{r['pista']}", int(r["ok"]))
            if "latencia_ms" in r:
                mlflow.log_metric(f"smoke_latencia_ms_{r['pista']}", r["latencia_ms"])

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19987.22it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

{'pista': 'mixta', 'run': 'scibert-A-natural-kd-lr3e-05-e4-s42', 'encoder': 'scibert', 'casos': 32, 'ok': True, 'predicciones_distintas': 0, 'max_dif_probabilidad': 0.0, 'latencia_ms': 9.893939125049656, 'checkpoint_sha256': '05fa410489932dc5da1893b834f31470a7d32eb331c47a6a1b19d550aca07fa1'}


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 30463.39it/s]
[transformers] BertModel LOAD REPORT from: /home/carlos/.cache/huggingface/hub/models--allenai--scibert_scivocab_uncased/snapshots/24f92d32b1bfb0bcaf9ab193ff3ad01e87732fc1
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archit

{'pista': 'sonnet', 'run': 'scibert-S-sonnet-lr3e-05-e4-s42', 'encoder': 'scibert', 'casos': 32, 'ok': True, 'predicciones_distintas': 0, 'max_dif_probabilidad': 0.0, 'latencia_ms': 5.705596281245562, 'checkpoint_sha256': 'fb66483eb857594f7f9f54f35742802709a84af25090e05afff3006a7f95a287'}


## Fase 8 — Análisis y manifiesto

Una tabla por pista (sin `a4-*` ni runs invalidados), ordenada por `val` propia; media ± desviación entre semillas de
los ganadores; bootstrap pareado (IC95% de la diferencia de macro-F1) y `resumen_part2.csv`.

In [13]:
def tabla_runs() -> pd.DataFrame:
    exp = mlflow.get_experiment_by_name(rm.EXPERIMENTO + SUFIJO)
    filas = []
    for r in mlflow.search_runs([exp.experiment_id], output_format="list"):
        t, m, p = r.data.tags, r.data.metrics, r.data.params
        nombre = t.get("mlflow.runName", "").rsplit("-", 2)[0]
        if r.info.status != "FINISHED" or "invalido" in t or nombre.startswith("a4-") or "val_f1_macro" not in m:
            continue
        hist = CLIENTE.get_metric_history(r.info.run_id, "f1_macro_val")
        filas.append({"pista": p.get("pista", "mixta"), "run": nombre, "run_id": r.info.run_id,
                      "lr": p.get("hp_lr"), "epocas_max": p.get("hp_epocas"), "mejor_epoca": max(hist, key=lambda h: h.value).step if hist else None,
                      **{k: m.get(k) for k in ("val_f1_macro", "val_mixta_f1_macro", "val_sonnet_f1_macro",
                                               "dev_haiku_f1_macro", "ref_gpt_f1_macro", "gpu_mem_pico_gb")}})
    return pd.DataFrame(filas)


TABLA = tabla_runs()
COLS = ["run", "lr", "epocas_max", "mejor_epoca", "val_f1_macro", "val_mixta_f1_macro", "val_sonnet_f1_macro",
        "dev_haiku_f1_macro", "ref_gpt_f1_macro", "gpu_mem_pico_gb"]
for pista in ("mixta", "sonnet"):
    print(f"=== pista {pista} (orden: val propia)")
    print(TABLA[TABLA.pista == pista].sort_values("val_f1_macro", ascending=False)[COLS].round(4).to_string(index=False))
EVIDENCIAS.mkdir(parents=True, exist_ok=True)
TABLA.drop(columns="run_id").sort_values(["pista", "val_f1_macro"], ascending=[True, False]).to_csv(EVIDENCIAS / "resumen_part2.csv", index=False)

=== pista mixta (orden: val propia)
                                  run    lr epocas_max  mejor_epoca  val_f1_macro  val_mixta_f1_macro  val_sonnet_f1_macro  dev_haiku_f1_macro  ref_gpt_f1_macro  gpu_mem_pico_gb
  scibert-A-natural-kd-lr3e-05-e4-s42 3e-05          4          4.0        0.7167              0.7167               0.5094              0.6697            0.5151            5.580
scibert-A-natural-kd-lr3e-05-e4-s1234 3e-05          4          4.0        0.7130              0.7130               0.4982              0.6128            0.4768            5.577
             scibert-A-natural-kd-s42 2e-05          6          5.0        0.7079              0.7079               0.5030              0.6007            0.4865            5.580
 scibert-A-natural-kd-lr1e-05-e10-s42 1e-05         10         10.0        0.7069              0.7069               0.4949              0.5984            0.5236            5.580
scibert-A-natural-kd-lr3e-05-e4-s2026 3e-05          4          4.0       

In [14]:
# media +- std entre semillas de los ganadores (mismo run sin el sufijo -s<semilla>)
for pista, b in MEJOR.items():
    grupo = TABLA[(TABLA.pista == pista) & (TABLA.run.str.replace(r"-s\d+$", "", regex=True) == b["stem"])]
    print(f"=== {pista}: {b['stem']} ({len(grupo)} semillas)")
    for col in ("val_f1_macro", "val_mixta_f1_macro", "val_sonnet_f1_macro", "dev_haiku_f1_macro", "ref_gpt_f1_macro"):
        if len(grupo) > 1:
            print(f"  {col}: {grupo[col].mean():.4f} +- {grupo[col].std():.4f}")

=== mixta: scibert-A-natural-kd-lr3e-05-e4 (3 semillas)
  val_f1_macro: 0.7113 +- 0.0065
  val_mixta_f1_macro: 0.7113 +- 0.0065
  val_sonnet_f1_macro: 0.5009 +- 0.0075
  dev_haiku_f1_macro: 0.6338 +- 0.0312
  ref_gpt_f1_macro: 0.4971 +- 0.0192
=== sonnet: scibert-S-sonnet-lr3e-05-e4 (3 semillas)
  val_f1_macro: 0.6667 +- 0.0133
  val_mixta_f1_macro: 0.5405 +- 0.0084
  val_sonnet_f1_macro: 0.6667 +- 0.0133
  dev_haiku_f1_macro: 0.5163 +- 0.0117
  ref_gpt_f1_macro: 0.6296 +- 0.0276


In [15]:
# Bootstrap pareado de la diferencia de macro-F1 sobre las citas comunes de ambos runs
ETIQ = {"val_mixta": {x["context_id"]: dominio.normalizar_etiqueta(x["etiqueta"]) for x in V},
        "val_sonnet": {x["context_id"]: dominio.normalizar_etiqueta(x["sonnet"]["label"]) for x in V_SONNET}}


def preds_de(run_id):
    return ex.leer_predicciones(Path(mlflow.get_run(run_id).data.tags["checkpoint"]).parent)


def comparar(run_a, run_b, conjunto):
    pa, pb = preds_de(run_a), preds_de(run_b)
    ids = sorted(set(pa) & set(pb) & set(ETIQ[conjunto]))
    return ex.bootstrap_diferencia([pa[c] for c in ids], [pb[c] for c in ids], [ETIQ[conjunto][c] for c in ids],
                                   n=N_BOOTSTRAP, semilla=2026)


NATURAL_P1 = CAND_M[0]["run"]
PARES = []
if NATURAL_P1:
    for pista, b in MEJOR.items():
        if b["run"] and b["run"] != NATURAL_P1:
            PARES += [(f"{pista}:{b['nombre']} vs M:natural(P1)", b["run"], NATURAL_P1, c) for c in ("val_mixta", "val_sonnet")]
if MEJOR["mixta"]["run"] and MEJOR["sonnet"]["run"]:
    PARES += [("S vs M", MEJOR["sonnet"]["run"], MEJOR["mixta"]["run"], c) for c in ("val_mixta", "val_sonnet")]
BOOT = []
for etiqueta, a, b, conjunto in PARES:
    try:
        r = comparar(a, b, conjunto)
    except Exception:
        FALLAS.append(f"bootstrap-{etiqueta}")
        print(f"[FALLO] bootstrap {etiqueta}\n{traceback.format_exc()}")
        continue
    BOOT.append({"comparacion": etiqueta, "conjunto": conjunto, **r})
    print(f"{etiqueta:55s} {conjunto:11s} dif={r['diferencia']:+.4f} IC95=[{r['ic95'][0]:+.4f}, {r['ic95'][1]:+.4f}] n={r['casos']}")
(EVIDENCIAS / "bootstrap_part2.json").write_text(json.dumps(BOOT, ensure_ascii=False, indent=2), encoding="utf-8")

mixta:scibert-A-natural-kd-lr3e-05-e4-s42 vs M:natural(P1) val_mixta   dif=+0.0223 IC95=[+0.0063, +0.0393] n=3379
mixta:scibert-A-natural-kd-lr3e-05-e4-s42 vs M:natural(P1) val_sonnet  dif=+0.0063 IC95=[-0.0081, +0.0211] n=3379
sonnet:scibert-S-sonnet-lr3e-05-e4-s42 vs M:natural(P1) val_mixta   dif=-0.1593 IC95=[-0.1865, -0.1317] n=3379
sonnet:scibert-S-sonnet-lr3e-05-e4-s42 vs M:natural(P1) val_sonnet  dif=+0.1705 IC95=[+0.1434, +0.1954] n=3379
S vs M                                                  val_mixta   dif=-0.1815 IC95=[-0.2088, -0.1544] n=3379
S vs M                                                  val_sonnet  dif=+0.1642 IC95=[+0.1356, +0.1910] n=3379


1709

In [16]:
DF = rm.exportar_manifiesto(EVIDENCIAS, rm.EXPERIMENTO + SUFIJO)
print(len(DF), "runs exportados a", EVIDENCIAS)
print("fallas:", FALLAS or "ninguna", f"| {(time.time() - T0) / 3600:.2f} h")

62 runs exportados a /home/carlos/microproyecto-local-citation/proyecto_de_grado/docs/evidencias_experimentos_104
fallas: ninguna | 7.53 h
